In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp
import csv
import pickle
from collections import deque
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
# 1. Configuration
WINDOW_SIZE = 15
INPUT_CSV = 'main_coords.csv' # filename
OUTPUT_CSV = 'tmp_data.csv'

print(f"Loading raw data from {INPUT_CSV}...")
# read the csv file into pandas dataframe
df_raw = pd.read_csv(INPUT_CSV)
df_raw['class'].value_counts()

In [ ]:
LABEL_COL = 'class'
# Drop the chaotic 'Clapping' class
df_raw = df_raw[df_raw[LABEL_COL] != 'Clapping']
print(f"Remaining classes to process: {df_raw[LABEL_COL].unique()}")

# Get all the feature columns (everything except the label)
feature_cols = [col for col in df_raw.columns if col != LABEL_COL]

temporal_dataset = []

# 2. Process each action separately
for action_name, group_df in df_raw.groupby(LABEL_COL):
    print(f"Processing '{action_name}' (Raw rows: {len(group_df)})...")
    
    action_data = group_df[feature_cols].values
    valid_windows_created = 0
    
    # 3. Slide the 15-frame window down the rows
    for i in range(len(action_data) - WINDOW_SIZE + 1):
        # Grab the 15 sequential rows. Shape is (15, 132)
        window = action_data[i : i + WINDOW_SIZE]
        
        # --- A. STANDARD STATISTICS (396 features) ---
        temporal_mean = np.mean(window, axis=0)
        temporal_std = np.std(window, axis=0)
        temporal_range = np.ptp(window, axis=0) 
        
        # --- B. EXTRACT SPECIFIC LANDMARKS FOR THE CUSTOM MATH ---
        # Landmark index * 4 = X coordinate. +1 = Y coordinate.
        l_shoulder_x, l_shoulder_y = window[:, 44], window[:, 45]
        r_shoulder_x, r_shoulder_y = window[:, 48], window[:, 49]
        l_wrist_x, l_wrist_y       = window[:, 60], window[:, 61]
        r_wrist_x, r_wrist_y       = window[:, 64], window[:, 65]
        
        # BUG FIX: Added r_hip_x which was missing in the previous version
        l_hip_x, l_hip_y           = window[:, 92], window[:, 93]
        r_hip_x, r_hip_y           = window[:, 96], window[:, 97] 
        
        l_ankle_x, l_ankle_y       = window[:, 108], window[:, 109]
        r_ankle_x, r_ankle_y       = window[:, 112], window[:, 113]

        # --- C. CALCULATE ENGINEERED FEATURES (For all 15 frames instantly) ---
        shoulder_width = np.sqrt((l_shoulder_x - r_shoulder_x)**2 + (l_shoulder_y - r_shoulder_y)**2) + 1e-6
        
        # 1. Define centralized points for the core body axis
        mid_shoulder_x = (l_shoulder_x + r_shoulder_x) / 2
        mid_shoulder_y = (l_shoulder_y + r_shoulder_y) / 2
        mid_hip_x = (l_hip_x + r_hip_x) / 2
        mid_hip_y = (l_hip_y + r_hip_y) / 2
        
        # 2. TORSO ANGLE (Solves Bending vs Sitting)
        # np.arctan2 calculates the angle of the spine in radians relative to the camera
        torso_angle = np.arctan2(mid_hip_y - mid_shoulder_y, mid_hip_x - mid_shoulder_x)
        
        # 3. Standard Geometric Ratios
        wrist_dist = np.sqrt((l_wrist_x - r_wrist_x)**2 + (l_wrist_y - r_wrist_y)**2) / shoulder_width
        ankle_dist = np.sqrt((l_ankle_x - r_ankle_x)**2 + (l_ankle_y - r_ankle_y)**2) / shoulder_width
        l_hand_raised = (l_wrist_y - l_shoulder_y) / shoulder_width
        r_hand_raised = (r_wrist_y - r_shoulder_y) / shoulder_width
        torso_height = np.sqrt((mid_shoulder_x - mid_hip_x)**2 + (mid_shoulder_y - mid_hip_y)**2) / shoulder_width
        
        # Stack them into a custom array of shape (15 frames, 7 features)
        custom_array = np.column_stack([
            wrist_dist, ankle_dist, l_hand_raised, r_hand_raised, 
            torso_height, torso_angle, mid_hip_y
        ])
        
        # --- D. CALCULATE CUSTOM STATISTICS (14 features) ---
        custom_means = np.mean(custom_array, axis=0)
        custom_variances = np.var(custom_array, axis=0)
        
        # --- E. COMBINE EVERYTHING (396 + 7 + 7 = 410 features) ---
        combined_features = np.concatenate([
            temporal_mean, temporal_std, temporal_range, 
            custom_means, custom_variances
        ])
        
        final_row = [action_name] + combined_features.tolist()
        temporal_dataset.append(final_row)
        valid_windows_created += 1
        
    print(f" -> Generated {valid_windows_created} temporal samples for '{action_name}'")

# 4. Save the new engineered dataset
# Note: We updated the range to 410 to match our new massive feature array!
new_columns = [LABEL_COL] + [f'feat_{i}' for i in range(410)]
df_temporal = pd.DataFrame(temporal_dataset, columns=new_columns)

df_temporal.to_csv(OUTPUT_CSV, index=False)
print(f"\nSuccess! Saved {len(df_temporal)} total temporal samples to {OUTPUT_CSV}")

In [ ]:
#read the temporal data
df = pd.read_csv('tmp_data.csv')
df.head()

In [ ]:
X = df.drop('class', axis=1)
y = df['class']
y

In [ ]:
#chronological splitting to avoid data leakage


# (Chronological Block Splitting):
X_train, X_test, y_train, y_test = [], [], [], []

# Group the temporal dataset by action class
for action in df['class'].unique():
    action_subset = df[df['class'] == action]
    
    # Calculate the 80% cutoff index
    split_idx = int(len(action_subset) * 0.8)
    
    # The first 80% of the timeline is Training
    train_block = action_subset.iloc[:split_idx]
    # The last 20% of the timeline is Testing
    test_block = action_subset.iloc[split_idx:]
    
    X_train.append(train_block.drop('class', axis=1))
    y_train.append(train_block['class'])
    
    X_test.append(test_block.drop('class', axis=1))
    y_test.append(test_block['class'])

# Combine the blocks back together
X_train = pd.concat(X_train)
y_train = pd.concat(y_train)
X_test = pd.concat(X_test)
y_test = pd.concat(y_test)

print(f"Training on first 80% of sequence: {len(X_train)} samples")
print(f"Testing on final 20% of sequence: {len(X_test)} samples")

In [ ]:
pipelines={
    'lr' : make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'rc' : make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf' : make_pipeline(StandardScaler(),RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)),
    'gb' : make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [ ]:
fit_models ={}
for algo, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algo] = model

In [ ]:
from sklearn.metrics import accuracy_score
import pickle


for algo, model in fit_models.items():
    yhat = model.predict(X_test)
    print(algo, accuracy_score(y_test, yhat))

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import matplotlib.pyplot as plt

In [ ]:
for algo, model in fit_models.items():
    print(f"\n{'='*50}")
    print(f"MODEL: {algo.upper()}")
    print(f"{'='*50}")

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    # Classification Report
    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=model.classes_
    )

    disp.plot(cmap='Blues')
    plt.title(f"Confusion Matrix - {algo.upper()}")
    plt.show()

In [ ]:
# 1. LOAD TRAINED RF PIPELINE
print("Loading Gradient Boost Classifier...")
with open('Human_actionV1.4.pkl', 'rb') as f:  # model
    model = pickle.load(f)

# 2. INITIALIZE MEDIAPIPE AND BUFFERS
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

WINDOW_SIZE = 15
pose_buffer = deque(maxlen=WINDOW_SIZE)
custom_buffer = deque(maxlen=WINDOW_SIZE) # ADDED: Your new custom feature buffer

# 3. START WEBCAM
cap = cv2.VideoCapture(0)
print("Webcam active. Click the video window and press 'q' to quit.")

with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7, model_complexity=0) as pose_model:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        frame = cv2.flip(frame, 1)
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose_model.process(image_rgb)
        
        action = "Buffering..."
        confidence = 0.0
        
        # --- EXACT CODE BLOCK STARTS HERE ---
        if results.pose_landmarks:
            pose = results.pose_landmarks.landmark
            
            # 1. Base Spatial Features
            spatial_features = [val for lm in pose for val in (lm.x, lm.y, lm.z, lm.visibility)]
            pose_buffer.append(spatial_features)
            
            # # --- HELPER ARRAYS FOR EASY MATH ---
            # l_shoulder = np.array([pose[11].x, pose[11].y])
            # r_shoulder = np.array([pose[12].x, pose[12].y])
            # l_wrist = np.array([pose[15].x, pose[15].y])
            # r_wrist = np.array([pose[16].x, pose[16].y])
            # l_hip = np.array([pose[23].x, pose[23].y])
            # r_hip = np.array([pose[24].x, pose[24].y])
            # l_ankle = np.array([pose[27].x, pose[27].y])
            # r_ankle = np.array([pose[28].x, pose[28].y])
            
            # # --- CALCULATE REFERENCE SCALE ---
            # shoulder_width = np.linalg.norm(l_shoulder - r_shoulder) + 1e-6
            
            # # --- ENGINEERED FEATURES (Appended per frame) ---
            # wrist_dist = np.linalg.norm(l_wrist - r_wrist) / shoulder_width
            # ankle_dist = np.linalg.norm(l_ankle - r_ankle) / shoulder_width
            # l_hand_raised = (pose[15].y - pose[11].y) / shoulder_width
            # r_hand_raised = (pose[16].y - pose[12].y) / shoulder_width
            # torso_height = np.linalg.norm(l_shoulder - l_hip) / shoulder_width
            
            # custom_frame_features = [
            #     wrist_dist, ankle_dist, l_hand_raised, r_hand_raised, 
            #     torso_height, pose[23].y, pose[24].y 
            # ]
            # custom_buffer.append(custom_frame_features)

            
            # --- HELPER ARRAYS FOR EASY MATH --- modified
            l_shoulder = np.array([pose[11].x, pose[11].y])
            r_shoulder = np.array([pose[12].x, pose[12].y])
            l_wrist = np.array([pose[15].x, pose[15].y])
            r_wrist = np.array([pose[16].x, pose[16].y])
            l_hip = np.array([pose[23].x, pose[23].y])
            r_hip = np.array([pose[24].x, pose[24].y])
            l_ankle = np.array([pose[27].x, pose[27].y])
            r_ankle = np.array([pose[28].x, pose[28].y])
            
            # --- CALCULATE REFERENCE SCALE ---
            shoulder_width = np.linalg.norm(l_shoulder - r_shoulder) + 1e-6
            
            # 1. Define centralized points for the core body axis
            mid_shoulder = (l_shoulder + r_shoulder) / 2.0
            mid_hip = (l_hip + r_hip) / 2.0
            
            # 2. TORSO ANGLE (Calculated on this single frame)
            # np.arctan2(y, x) -> using the numpy arrays we just created
            torso_angle = np.arctan2(mid_hip[1] - mid_shoulder[1], mid_hip[0] - mid_shoulder[0])
            
            # 3. ENGINEERED FEATURES (Appended per frame)
            wrist_dist = np.linalg.norm(l_wrist - r_wrist) / shoulder_width
            ankle_dist = np.linalg.norm(l_ankle - r_ankle) / shoulder_width
            l_hand_raised = (pose[15].y - pose[11].y) / shoulder_width
            r_hand_raised = (pose[16].y - pose[12].y) / shoulder_width
            torso_height = np.linalg.norm(mid_shoulder - mid_hip) / shoulder_width
            
            # 4. Assemble the 7 custom features EXACTLY matching training order
            custom_frame_features = [
                wrist_dist, 
                ankle_dist, 
                l_hand_raised, 
                r_hand_raised, 
                torso_height, 
                torso_angle, 
                mid_hip[1] # mid_hip_y
            ]
            custom_buffer.append(custom_frame_features)
            
            # --- INFERENCE BLOCK ---
            if len(pose_buffer) == WINDOW_SIZE:
                try:
                    buffer_array = np.array(pose_buffer)
                    temporal_mean = np.mean(buffer_array, axis=0)
                    temporal_std = np.std(buffer_array, axis=0)
                    temporal_range = np.ptp(buffer_array, axis=0)
                    
                    custom_array = np.array(custom_buffer)
                    custom_means = np.mean(custom_array, axis=0)
                    custom_variances = np.var(custom_array, axis=0)
                    
                    combined_features = np.concatenate([
                        temporal_mean, temporal_std, temporal_range, 
                        custom_means, custom_variances
                    ])
                    
                    X = pd.DataFrame([combined_features])
                    
                    # The actual prediction execution for Ridge Classifier
                    # action = model.predict(X)[0]
                    # decision_scores = model.decision_function(X)
                    # confidence = (np.max(decision_scores) / (np.sum(np.abs(decision_scores)) + 1e-6)) * 100 
                    
                    # The actual prediction execution for Random Forest
                    action = model.predict(X)[0]
                    confidence = np.max(model.predict_proba(X)) * 100                           
                    
                except Exception as e:
                    action = "Shape Error!"
                    print(f"DEBUG: {e}")
            
            mp_drawing.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        # --- EXACT CODE BLOCK ENDS HERE ---

        # Draw UI
        cv2.rectangle(frame, (10, 10), (450, 100), (0, 0, 0), -1)
        cv2.putText(frame, f"ACTION: {action}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(frame, f"CONF: {confidence:.1f}%", (20, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 1)

        cv2.imshow('Live Testing', frame)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [ ]:
with open('Human_actionV1.4.pkl', 'wb') as f:
    pickle.dump(fit_models['rf'], f)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("Loading Gradient Boost Classifier...")
with open('Human_actionV1.3.pkl', 'rb') as f:  # model
    model = pickle.load(f)

# 1. Get the raw predictions from your trained model (Gradient Boosting or Random Forest)
y_pred_raw = model.predict(X_test)
y_pred_filtered = y_pred_raw.copy()

# 2. Extract the wrist variance column (feat_403) from your test data
# Handle both Pandas DataFrame or raw Numpy Array formats just in case
if isinstance(X_test, pd.DataFrame):
    wrist_variances = X_test['feat_403'].values
else:
    wrist_variances = X_test[:, 403]

# 3. Set your threshold (You can tweak this number in Jupyter to find the perfect balance)
JITTER_THRESHOLD = 0.005

# 4. Find all rows where the model guessed "Clapping" BUT the hands weren't moving enough
override_mask = (y_pred_filtered == 'Clapping') & (wrist_variances < JITTER_THRESHOLD)

# 5. Apply the override! Change those false positives to "Standing"
y_pred_filtered[override_mask] = 'Standing'

# Print the exact number of predictions the filter corrected
overridden_count = np.sum(override_mask)
print(f"✅ Filter applied! Changed {overridden_count} false 'Clapping' predictions to 'Standing'.\n")

# 6. Evaluate the new, filtered results
print("=== NEW CLASSIFICATION REPORT (WITH FILTER) ===")
print(classification_report(y_test, y_pred_filtered))

# 7. Plot the new Confusion Matrix to visually confirm the fix
cm = confusion_matrix(y_test, y_pred_filtered, labels=model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, cmap='Blues')
plt.title(f"Confusion Matrix (Filtered - Thresh: {JITTER_THRESHOLD})")
plt.show()